In [0]:
# =============================================================================
# Notebook  : 00_check_new_files
# Location  : /HGI-Lakehouse-Pipeline/00_Orchestration/00_check_new_files
# Purpose   : GATE TASK — checks whether new files arrived in the incremental
#             landing zone for BOTH Salesforce and BigQuery.
#
#             Sets a Databricks task value:
#               dbutils.jobs.taskValues.set("new_files_found", "true")
#               or
#               dbutils.jobs.taskValues.set("new_files_found", "false")
#
#             Downstream tasks in master_jobs.yml use run_if:
#               tasks['check_new_files'].values['new_files_found'] == 'true'
#
#             This means if NO new files arrived:
#               - Bronze tasks are SKIPPED automatically
#               - Silver tasks are SKIPPED automatically
#               - The job finishes immediately with no wasted compute
#
# HOW IT WORKS:
#   1. Scans S3 landing-zone/salesforce/{customer_id}/*/incremental/
#   2. Scans S3 landing-zone/bigquery/{customer_id}/events/incremental/
#   3. Uses Auto Loader's checkpoint metadata to find UNPROCESSED files only
#   4. If any unprocessed files exist → new_files_found = "true"
#   5. If no unprocessed files → new_files_found = "false"
#
# Serverless: works on Serverless (uses dbutils.fs.ls, no spark.conf issues)
# Job Compute: same code, same result
# =============================================================================

In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id", "")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

from datetime import datetime

print(f"=== Gate: Check New Files ===")
print(f"  customer_id : {customer_id}")
print(f"  Checked at  : {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  S3 root     : {landing_root}")

In [0]:
# CELL 3 ── Helper functions (optimized: single-pass scan + parallel-ready)
from concurrent.futures import ThreadPoolExecutor, as_completed


def scan_tree(path: str, since_ms: float = 0.0,
              depth: int = 0, max_depth: int = 5):
    """
    Single-pass recursive scan.  Returns (all_count, new_count, new_paths)
    in ONE tree walk — replaces the old count_all_files_in_tree,
    count_new_files_in_tree, and _collect_new_file_paths.
    """
    if depth > max_depth:
        return 0, 0, []
    all_count = 0
    new_count = 0
    new_paths = []
    try:
        for entry in dbutils.fs.ls(path):
            name = entry.name
            if name.endswith(".parquet") or name.endswith(".snappy.parquet"):
                all_count += 1
                if entry.modificationTime > since_ms:
                    new_count += 1
                    new_paths.append(entry.path)
            elif entry.isDir() and not name.startswith("_"):
                a, n, p = scan_tree(entry.path, since_ms, depth + 1, max_depth)
                all_count += a
                new_count += n
                new_paths.extend(p)
    except Exception:
        pass
    return all_count, new_count, new_paths


def get_latest_checkpoint_commit_ms(layer: str, source_sys: str, obj_name: str) -> float:
    """
    Returns the modification time (epoch ms) of the latest Auto Loader
    checkpoint commit file.  Returns 0.0 if no checkpoint exists (first run).
    """
    chk = checkpoint_path(layer, source_sys, customer_id, obj_name)
    try:
        commits = dbutils.fs.ls(chk.rstrip("/") + "/commits/")
        if not commits:
            return 0.0
        return max(c.modificationTime for c in commits)
    except Exception:
        return 0.0


def verify_sf_new_data(obj_name: str, new_paths: list) -> int:
    """
    Content-aware verification for Salesforce landing-zone files.

    Accepts pre-collected new file paths (from scan_tree) so the tree
    is NOT walked again.  Same logic as before:
      - Reads parquet files and checks SystemModstamp vs bronze
      - Catches 100%-duplicate files created by SF ingestion

    Returns:
        >0  = count of genuinely new/updated records
         0  = all records are duplicates (no pipeline run needed)
        -1  = could not verify (error / missing column) → assume new data
    """
    if not new_paths:
        return 0

    try:
        # Read the "new" files
        df = spark.read.parquet(*new_paths)
        if len(df.head(1)) == 0:
            return 0   # files exist but contain 0 rows

        # Find the best timestamp column (raw SF column names)
        ts_col = next(
            (c for c in ["SystemModstamp", "LastModifiedDate", "CreatedDate"]
             if c in df.columns),
            None
        )
        if ts_col is None:
            return -1  # no timestamp column → can't verify, assume new

        max_file_ts = df.selectExpr(
            f"MAX(CAST(`{ts_col}` AS TIMESTAMP))"
        ).collect()[0][0]
        if max_file_ts is None:
            return 0   # all timestamps NULL → no useful data

        # Compare against bronze table's max data_timestamp
        bronze_tbl = bronze_table(customer_id, obj_name)
        max_bronze_ts = spark.sql(
            f"SELECT MAX(data_timestamp) FROM {bronze_tbl}"
        ).collect()[0][0]

        if max_bronze_ts is None:
            # Bronze table empty → first load → all records are new
            return df.count()

        # If file's latest record is newer → count how many are new
        if max_file_ts > max_bronze_ts:
            return df.filter(
                f"CAST(`{ts_col}` AS TIMESTAMP) > CAST('{max_bronze_ts}' AS TIMESTAMP)"
            ).count()
        else:
            return 0   # all records ≤ bronze max → duplicates

    except Exception as e:
        print(f"    ⚠ verify_sf_new_data({obj_name}): {e}")
        return -1      # error → can't verify, assume new data to be safe

In [0]:
# CELL 4 ── Check Salesforce + BigQuery in PARALLEL (single-pass + threaded)
SF_OBJECTS = ["account", "contact", "opportunity", "task",
              "campaign", "campaignmember", "user"]


def _check_sf_object(obj):
    """Scan + verify a single SF object. Thread-safe."""
    path     = landing_path("salesforce", customer_id, obj, "incremental")
    chkpt_ms = get_latest_checkpoint_commit_ms("bronze", "salesforce", obj)
    all_count, new_count, new_paths = scan_tree(path, since_ms=chkpt_ms)

    verified_new = 0
    if new_count > 0:
        verified_new = verify_sf_new_data(obj, new_paths)
        if verified_new == 0:
            new_count = 0
        # verified_new == -1 → keep original new_count (safe fallback)

    return obj, all_count, new_count, chkpt_ms, verified_new


def _check_bq():
    """Scan BigQuery events landing zone. Thread-safe."""
    path     = landing_path("bigquery", customer_id, "events", "incremental")
    chkpt_ms = get_latest_checkpoint_commit_ms("bronze", "bigquery", "events")
    all_count, new_count, _ = scan_tree(path, since_ms=chkpt_ms)
    return all_count, new_count, chkpt_ms


# ── Run ALL checks in parallel (7 SF objects + 1 BQ = 8 threads) ─────────────
print(f"\n── Salesforce incremental files ──")
sf_new_total = 0
sf_all_total = 0
sf_details   = {}

with ThreadPoolExecutor(max_workers=len(SF_OBJECTS) + 1) as pool:
    sf_futures = {pool.submit(_check_sf_object, obj): obj for obj in SF_OBJECTS}
    bq_future  = pool.submit(_check_bq)

    # Collect SF results as they complete
    sf_results = {}
    for future in as_completed(sf_futures):
        obj, all_count, new_count, chkpt_ms, verified_new = future.result()
        sf_results[obj] = (all_count, new_count, chkpt_ms, verified_new)

    # Collect BQ result (may already be done)
    bq_all_total, bq_new_total, bq_chkpt_ms = bq_future.result()

# ── Print SF results in original order ────────────────────────────────────────
for obj in SF_OBJECTS:
    all_count, new_count, chkpt_ms, verified_new = sf_results[obj]
    sf_details[obj] = {"all": all_count, "new": new_count, "chkpt_ms": chkpt_ms,
                       "verified": verified_new}
    sf_all_total += all_count
    sf_new_total += new_count

    chkpt_info = ("no checkpoint (first run)" if chkpt_ms == 0.0
                  else f"checkpoint @ {datetime.utcfromtimestamp(chkpt_ms / 1000).strftime('%Y-%m-%d %H:%M')}")

    if new_count > 0:
        icon = "❗ NEW"
    elif verified_new == 0 and all_count > 0:
        icon = "─ dup"   # files exist but verified as duplicates
    else:
        icon = "─"       # no files at all
    print(f"  {icon:<7} {obj:<20} {new_count:>5} new / {all_count:>5} total  |  {chkpt_info}")
    if verified_new == 0 and sf_details[obj]["all"] > 0 and sf_details[obj]["verified"] == 0:
        print(f"          └─ content check: all records already in bronze (duplicates)")

In [0]:
# CELL 5 ── Print BigQuery results (already computed in parallel above)
print(f"\n── BigQuery incremental files ──")
chkpt_info = ("no checkpoint (first run)" if bq_chkpt_ms == 0.0
              else f"checkpoint @ {datetime.utcfromtimestamp(bq_chkpt_ms / 1000).strftime('%Y-%m-%d %H:%M')}")
icon = "?❗" if bq_new_total > 0 else "─"
print(f"  {icon}  {'events':<20} {bq_new_total:>5} new / {bq_all_total:>5} total  |  {chkpt_info}")

In [0]:
# CELL 6 ── Summary: which objects have truly NEW unprocessed files
print(f"\n── New-file summary (checkpoint-aware) ──")

for obj in SF_OBJECTS:
    d    = sf_details[obj]
    icon = "✅ NEW" if d["new"] > 0 else "⏸  none"
    print(f"  {icon}  salesforce/{obj:<20}  {d['new']:>5} new files")

bq_icon = "✅ NEW" if bq_new_total > 0 else "⏸  none"
print(f"  {bq_icon}  bigquery/events            {bq_new_total:>5} new files")

In [0]:
# CELL 7 ── (reserved — logic moved to checkpoint-aware cells above)

In [0]:
# CELL 8 ── Final gate decision  (based on NEW files only, not all files)
total_new_files = sf_new_total + bq_new_total
new_files_found = total_new_files > 0

# Keep sf_total / bq_total aliases for Cell 9 task-value compatibility
sf_total = sf_new_total
bq_total = bq_new_total

print(f"\n── Gate Decision ──")
print(f"  SF files in landing zone  : {sf_all_total:,} total, {sf_new_total:,} NEW")
print(f"  BQ files in landing zone  : {bq_all_total:,} total, {bq_new_total:,} NEW")
print(f"  Files to process          : {total_new_files:,}")
print()
if new_files_found:
    print(f"  ✅ NEW FILES FOUND → Setting new_files_found = 'true'")
    print(f"     All downstream bronze + silver tasks will RUN")
else:
    print(f"  ⏸  NO NEW FILES → Setting new_files_found = 'false'")
    print(f"     All downstream tasks will be SKIPPED (saves cost)")

In [0]:
# CELL 9 ── Set task value — this is what downstream run_if conditions read
# This is a Databricks-native feature: dbutils.jobs.taskValues
# Works in Jobs only (raises error in interactive mode — handled below)
try:
    dbutils.jobs.taskValues.set(key="new_files_found", value=str(new_files_found).lower())
    dbutils.jobs.taskValues.set(key="sf_files_count",  value=str(sf_total))
    dbutils.jobs.taskValues.set(key="bq_files_count",  value=str(bq_total))
    print(f"\n  Task values set:")
    print(f"    new_files_found = '{str(new_files_found).lower()}'")
    print(f"    sf_files_count  = '{sf_total}'")
    print(f"    bq_files_count  = '{bq_total}'")
except Exception as e:
    # Running interactively (not inside a job) — task values not available
    print(f"\n  NOTE: Running interactively — task values not set (only works inside a Job)")
    print(f"  new_files_found would be: {str(new_files_found).lower()}")

In [0]:
# CELL 10 ── Exit with the result (used by the calling job for run_if)
dbutils.notebook.exit(str(new_files_found).lower())   # "true" or "false"